Cải tiến feature:
1. **Holiday indicator** — ngày lễ liên bang Mỹ + ngày đặc biệt NYC
2. **COVID lockdown flag** — flag giai đoạn lockdown 2020-2021
3. **Zone-level demand normalization** — chuẩn hóa demand theo từng zone
4. **Extended XGBoost grid** — thêm `colsample_bytree`, `min_child_weight`

In [1]:
%pip install holidays pyarrow scikit-learn xgboost statsmodels


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, numpy as np, pandas as pd
import holidays
from datetime import date
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import StructType, StructField
from xgboost.spark import SparkXGBRegressor

spark = (
    SparkSession.builder
    .appName("FeatureEnhancement_VirtualCluster")
    .master("spark://master:7077")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .config("spark.driver.memory", "3g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.executor.memory", "5g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.executor.cores", "3")
    .config("spark.cores.max", "9")
    .config("spark.sql.shuffle.partitions", "48")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark ready:", spark.version)


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9d267863-544c-4e57-a096-229f04cdb773;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 106ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

Spark ready: 3.5.0


In [3]:
FEATURE_PATH = "/user/data/feature_engineering/demand_prediction_features"
TARGET_COL   = "pickup_demand"
MAPE_THRESH  = 5

df_all = spark.read.parquet(FEATURE_PATH)
if "split_local" not in df_all.columns and "split" in df_all.columns:
    df_all = df_all.withColumn("split_local", F.col("split"))

print("Schema:", df_all.columns)
print("Total rows:", df_all.count())


Schema: ['PULocationID', 'pickup_bin_60m', 'pickup_demand', 'cluster_id', 'hour', 'dow', 'month', 'is_weekend', 'lag_3', 'lag_168', 'lag_6', 'roll_mean_6', 'roll_mean_24', 'roll_std_24', 'rn', 'n_zone', 'train_cut', 'split', 'split_local']


Total rows: 12008760


In [4]:
# === Build holiday lookup: 2019-2026 ===
us_holidays = holidays.UnitedStates(years=range(2019, 2027))

# NYC-specific additions
nyc_special = {
    date(2020, 1, 1), date(2021, 1, 1), date(2022, 1, 1),
    date(2023, 1, 1), date(2024, 1, 1), date(2025, 1, 1),
    date(2019, 12, 31), date(2020, 12, 31), date(2021, 12, 31),
    date(2022, 12, 31), date(2023, 12, 31), date(2024, 12, 31),
    # NYC Marathon (first Sunday Nov)
    date(2019, 11, 3), date(2021, 11, 7), date(2022, 11, 6),
    date(2023, 11, 5), date(2024, 11, 3),
}

holiday_dates = set(us_holidays.keys()) | nyc_special

# === COVID lockdown periods NYC ===
# Phase 1 lockdown: 2020-03-22 to 2020-06-07
# Partial restrictions: 2020-06-08 to 2021-05-19
covid_lockdown_ranges = [
    (date(2020, 3, 22), date(2020, 6, 7)),   # full lockdown
]
covid_partial_ranges = [
    (date(2020, 1, 1), date(2020, 3, 21)),   # pre-lockdown uncertainty
    (date(2020, 6, 8), date(2021, 5, 19)),   # partial restrictions
]

# Build date-level lookup DataFrame and broadcast
date_rows = []
for d in pd.date_range("2019-01-01", "2026-12-31"):
    dt = d.date()
    is_hol = int(dt in holiday_dates)
    is_covid_lock = int(any(s <= dt <= e for s, e in covid_lockdown_ranges))
    is_covid_part = int(any(s <= dt <= e for s, e in covid_partial_ranges))
    date_rows.append((d.strftime("%Y-%m-%d"), is_hol, is_covid_lock, is_covid_part))

date_schema = "date_str STRING, is_holiday INT, is_covid_lockdown INT, is_covid_partial INT"
date_lookup_df = spark.createDataFrame(date_rows, schema=date_schema)

holiday_count = sum(1 for _, h, _, _ in date_rows if h)
lockdown_count = sum(1 for _, _, l, _ in date_rows if l)
print(f"Holiday days flagged: {holiday_count}")
print(f"Full lockdown days:   {lockdown_count}")

# Join to main DataFrame
df_flagged = (
    df_all
    .withColumn("date_str", F.date_format(F.col("pickup_bin_60m"), "yyyy-MM-dd"))
    .join(F.broadcast(date_lookup_df), on="date_str", how="left")
    .fillna({"is_holiday": 0, "is_covid_lockdown": 0, "is_covid_partial": 0})
    .withColumn("is_holiday", F.col("is_holiday").cast(DoubleType()))
    .withColumn("is_covid_lockdown", F.col("is_covid_lockdown").cast(DoubleType()))
    .withColumn("is_covid_partial", F.col("is_covid_partial").cast(DoubleType()))
    .drop("date_str")
)

print("Flagged schema sample:")
df_flagged.select("pickup_bin_60m", "is_holiday", "is_covid_lockdown", "is_covid_partial").show(5)


Holiday days flagged: 106
Full lockdown days:   78
Flagged schema sample:


+-------------------+----------+-----------------+----------------+
|     pickup_bin_60m|is_holiday|is_covid_lockdown|is_covid_partial|
+-------------------+----------+-----------------+----------------+
|2020-01-08 00:00:00|       0.0|              0.0|             1.0|
|2020-01-08 01:00:00|       0.0|              0.0|             1.0|
|2020-01-08 02:00:00|       0.0|              0.0|             1.0|
|2020-01-08 03:00:00|       0.0|              0.0|             1.0|
|2020-01-08 04:00:00|       0.0|              0.0|             1.0|
+-------------------+----------+-----------------+----------------+
only showing top 5 rows



In [5]:
# ---- Add Holt-Winters forecast feature (needed before normalization) ----
# hw_forecast is NOT stored in HDFS, must be recomputed here
import numpy as np
from pyspark.sql.types import StructType, StructField, DoubleType as DT

HW_SEASONAL_PERIODS = 24
HW_MAX_TRAIN = HW_SEASONAL_PERIODS * 52
HW_MIN_ROWS  = HW_SEASONAL_PERIODS * 3
HW_MIN_TRAIN = HW_SEASONAL_PERIODS * 2

def add_hw_forecast(pdf):
    pdf = pdf.sort_values("pickup_bin_60m").reset_index(drop=True)
    y = pdf["pickup_demand"].astype(float).to_numpy()
    if len(y) < HW_MIN_ROWS:
        pdf["hw_forecast"] = y; return pdf
    train_mask = pdf["split_local"] == "train"
    y_train_full = y[train_mask.to_numpy()]
    if len(y_train_full) < HW_MIN_TRAIN:
        pdf["hw_forecast"] = y; return pdf
    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        y_fit = y_train_full[-HW_MAX_TRAIN:]
        n_older = len(y_train_full) - len(y_fit)
        n_test  = int((~train_mask).sum())
        hw = ExponentialSmoothing(y_fit, trend="add", seasonal="add",
                                   seasonal_periods=HW_SEASONAL_PERIODS,
                                   initialization_method="estimated")
        fitted = hw.fit(optimized=True)
        in_sample = np.asarray(fitted.fittedvalues)
        train_fc = np.concatenate([np.full(n_older, in_sample[0]), in_sample]) if n_older > 0 else in_sample
        test_fc  = np.asarray(fitted.forecast(n_test)) if n_test > 0 else np.array([])
        forecast = np.concatenate([train_fc, test_fc]) if n_test > 0 else train_fc
        pdf["hw_forecast"] = np.maximum(forecast, 0.0)
    except Exception:
        pdf["hw_forecast"] = y
    return pdf

hw_schema = StructType(df_flagged.schema.fields + [StructField("hw_forecast", DT(), True)])
print("Computing Holt-Winters forecast feature... (takes several minutes)")
df_flagged = df_flagged.groupBy("PULocationID").applyInPandas(add_hw_forecast, schema=hw_schema).cache()
print("HW forecast done. Sample:")
df_flagged.select("PULocationID", "pickup_bin_60m", "pickup_demand", "hw_forecast").show(5)


Computing Holt-Winters forecast feature... (takes several minutes)
HW forecast done. Sample:


+------------+-------------------+-------------+-----------+
|PULocationID|     pickup_bin_60m|pickup_demand|hw_forecast|
+------------+-------------------+-------------+-----------+
|          18|2020-01-08 00:00:00|          0.0|        0.0|
|          18|2020-01-08 01:00:00|          0.0|        0.0|
|          18|2020-01-08 02:00:00|          0.0|        0.0|
|          18|2020-01-08 03:00:00|          0.0|        0.0|
|          18|2020-01-08 04:00:00|          0.0|        0.0|
+------------+-------------------+-------------+-----------+
only showing top 5 rows



In [6]:
# Compute per-zone mean and std from TRAINING data only (prevent leakage)
zone_stats = (
    df_flagged
    .filter(F.col("split_local") == "train")
    .filter(F.col(TARGET_COL) > 0)
    .groupBy("PULocationID")
    .agg(
        F.avg(TARGET_COL).alias("zone_mean"),
        F.stddev_pop(TARGET_COL).alias("zone_std"),
    )
    .withColumn("zone_std", F.when(F.col("zone_std") < 0.01, F.lit(1.0)).otherwise(F.col("zone_std")))
)

print("Zone stats sample:")
zone_stats.orderBy(F.col("zone_mean").desc()).show(10)

# Join zone stats and create normalized demand target
df_enhanced = (
    df_flagged
    .join(F.broadcast(zone_stats), on="PULocationID", how="left")
    .fillna({"zone_mean": 1.0, "zone_std": 1.0})
    # demand_norm = (demand - zone_mean) / zone_std  → z-score within zone
    .withColumn(
        "demand_norm",
        ((F.col(TARGET_COL) - F.col("zone_mean")) / F.col("zone_std")).cast(DoubleType())
    )
    # Also normalize lag/roll features
    .withColumn("lag_3_norm",        ((F.col("lag_3")        - F.col("zone_mean")) / F.col("zone_std")))
    .withColumn("lag_6_norm",        ((F.col("lag_6")        - F.col("zone_mean")) / F.col("zone_std")))
    .withColumn("lag_168_norm",      ((F.col("lag_168")      - F.col("zone_mean")) / F.col("zone_std")))
    .withColumn("roll_mean_6_norm",  ((F.col("roll_mean_6")  - F.col("zone_mean")) / F.col("zone_std")))
    .withColumn("roll_mean_24_norm", ((F.col("roll_mean_24") - F.col("zone_mean")) / F.col("zone_std")))
    .withColumn("hw_norm",           ((F.col("hw_forecast")  - F.col("zone_mean")) / F.col("zone_std")))
    .cache()
)

print(f"Enhanced rows: {df_enhanced.count():,}")


Zone stats sample:


+------------+------------------+------------------+
|PULocationID|         zone_mean|          zone_std|
+------------+------------------+------------------+
|          48|  64.7467880085653| 38.85598534009256|
|         263| 61.84818007662835|42.392657280759046|
|         107| 58.62454686690834| 42.26919719887817|
|         100| 57.97165820642978| 36.54137434993833|
|          90| 54.91594076655052| 38.71290950854381|
|         229| 53.89668615984405|42.025276074933856|
|          79| 52.13030888030888|35.332991357695306|
|         186| 51.96127562642369| 37.58932691795622|
|         249|50.925824175824175| 36.21944198632254|
|         164|50.762824048538334| 40.35843278316334|
+------------+------------------+------------------+
only showing top 10 rows



26/05/08 15:07:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Enhanced rows: 12,008,760


Khi train trên `demand_norm`, prediction của model cũng là normalized.
Để tính metric thực, cần **denormalize** lại: `pred_actual = pred_norm * zone_std + zone_mean`

In [7]:
# Feature columns for enhanced model (normalized + new flags)
NORM_TARGET = "demand_norm"  # train on normalized demand

feature_cols_enhanced = [
    "hour", "dow", "month", "is_weekend",
    "is_holiday", "is_covid_lockdown", "is_covid_partial",
    "lag_3_norm", "lag_6_norm", "lag_168_norm",
    "roll_mean_6_norm", "roll_mean_24_norm", "roll_std_24",
    "hw_norm", "cluster_id",
]

train_enhanced = df_enhanced.filter(F.col("split_local") == "train").filter(F.col(TARGET_COL) > 0)
test_enhanced  = df_enhanced.filter(F.col("split_local") == "test")

print(f"Train (non-zero): {train_enhanced.count():,}")
print(f"Test (all):       {test_enhanced.count():,}")


Train (non-zero): 249,461
Test (all):       3,602,628


In [8]:
# Evaluation helper — denormalize before computing metrics
results_enhanced = []

def evaluate_enhanced(model_name, pred_df):
    # Denormalize: pred_actual = pred_norm * zone_std + zone_mean
    eval_df = (
        pred_df
        .withColumn(
            "pred_actual",
            F.greatest(
                F.col("prediction") * F.col("zone_std") + F.col("zone_mean"),
                F.lit(0.0)
            )
        )
    )
    nonzero = eval_df.filter(F.col(TARGET_COL) > 0)
    mae_v  = float(nonzero.agg(F.avg(F.abs(F.col(TARGET_COL) - F.col("pred_actual")))).first()[0])
    rmse_v = float(nonzero.agg(F.sqrt(F.avg(F.pow(F.col(TARGET_COL) - F.col("pred_actual"), 2)))).first()[0])
    wape_v = float(nonzero.agg(
        F.sum(F.abs(F.col(TARGET_COL) - F.col("pred_actual"))) / F.sum(F.col(TARGET_COL)) * 100.0
    ).first()[0])
    high = nonzero.filter(F.col(TARGET_COL) >= MAPE_THRESH)
    mape_v = float(high.agg(
        F.avg(F.abs(F.col(TARGET_COL) - F.col("pred_actual")) / F.col(TARGET_COL)) * 100.0
    ).first()[0] or 0)
    # R2 on normalized space
    ss_res = float(nonzero.agg(F.sum(F.pow(F.col(NORM_TARGET) - F.col("prediction"), 2))).first()[0])
    ss_tot = float(nonzero.agg(F.sum(F.pow(F.col(NORM_TARGET) - F.avg(F.col(NORM_TARGET)).over(
        __import__('pyspark.sql').sql.Window.partitionBy(F.lit(1))), 2))).first()[0]) if False else None
    r2_eval_obj = __import__('pyspark.ml.evaluation', fromlist=['RegressionEvaluator']).RegressionEvaluator(
        labelCol=NORM_TARGET, predictionCol="prediction", metricName="r2")
    r2_v = float(r2_eval_obj.evaluate(nonzero))
    print(f"\n=== {model_name} (Enhanced) ===")
    print(f"  MAE      (actual scale): {mae_v:.4f}")
    print(f"  RMSE     (actual scale): {rmse_v:.4f}")
    print(f"  WAPE     (actual scale): {wape_v:.2f}%")
    print(f"  MAPE(>=5)(actual scale): {mape_v:.2f}%")
    print(f"  R²       (norm  space) : {r2_v:.4f}")
    row = {"model": model_name, "MAE": mae_v, "RMSE": rmse_v,
           "WAPE": wape_v, "MAPE(>=5)": mape_v, "R2_norm": r2_v}
    results_enhanced.append(row)
    return row

assembler_enh = VectorAssembler(
    inputCols=feature_cols_enhanced, outputCol="features", handleInvalid="skip")


In [9]:
xgb_enh = SparkXGBRegressor(
    features_col="features",
    label_col=NORM_TARGET,
    prediction_col="prediction",
    num_workers=3,
    objective="reg:squarederror",
    device="cpu",
)

xgb_enh_pipeline = Pipeline(stages=[assembler_enh, xgb_enh])

# Extended grid: added colsample_bytree and min_child_weight
xgb_enh_grid = (
    ParamGridBuilder()
    .addGrid(xgb_enh.max_depth,         [5, 7])
    .addGrid(xgb_enh.learning_rate,     [0.05, 0.1])
    .addGrid(xgb_enh.n_estimators,      [80, 150])
    .addGrid(xgb_enh.subsample,         [0.7, 0.9])
    .addGrid(xgb_enh.colsample_bytree,  [0.7, 1.0])
    .build()
)

print(f"XGBoost extended grid: {len(xgb_enh_grid)} combinations × 3 folds")

cv_xgb_enh = CrossValidator(
    estimator=xgb_enh_pipeline,
    estimatorParamMaps=xgb_enh_grid,
    evaluator=RegressionEvaluator(
        labelCol=NORM_TARGET, predictionCol="prediction", metricName="rmse"),
    numFolds=3,
    parallelism=2,
    seed=42,
)

print("Fitting XGBoost Enhanced... (may take 20-50 mins)")
cv_xgb_enh_model = cv_xgb_enh.fit(train_enhanced)

best_idx = int(np.argmin(cv_xgb_enh_model.avgMetrics))
best_params = cv_xgb_enh_model.getEstimatorParamMaps()[best_idx]
print("\nBest XGBoost Enhanced params:")
for k, v in best_params.items():
    print(f"  {k.name}: {v}")

xgb_enh_pred = cv_xgb_enh_model.transform(test_enhanced)
evaluate_enhanced("xgboost_enhanced", xgb_enh_pred)


XGBoost extended grid: 32 combinations × 3 folds
Fitting XGBoost Enhanced... (may take 20-50 mins)


2026-05-08 15:08:05,480 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'objective': 'reg:squarederror', 'subsample': 0.7, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 80}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-08 15:08:05,483 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'colsample_bytree': 0.7, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'objective': 'reg:squarederror', 'subsample': 0.7, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 80}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
[15:08:10] [0]	training-rmse:0.99776/ 3][Stage 48:>                 (0 + 3) / 3]
[15:08:10] [0]	training-rmse:0.99078
[15:08:10] [1]	training-rmse:0.99583
[15:08:10] [1]	training-rmse:0.98195
[15:08:10] [2]	training-rmse:0.98711
[15:08:10] [2]	trai


Best XGBoost Enhanced params:
  max_depth: 7
  learning_rate: 0.1
  n_estimators: 150
  subsample: 0.9
  colsample_bytree: 0.7



=== xgboost_enhanced (Enhanced) ===
  MAE      (actual scale): 6.1193
  RMSE     (actual scale): 12.1482
  WAPE     (actual scale): 38.39%
  MAPE(>=5)(actual scale): 56.73%
  R²       (norm  space) : 0.0217


{'model': 'xgboost_enhanced',
 'MAE': 6.119282070033551,
 'RMSE': 12.148173329701425,
 'WAPE': 38.387733232030804,
 'MAPE(>=5)': 56.730696628235386,
 'R2_norm': 0.021680539992476877}

In [10]:
summary_enh = pd.DataFrame(results_enhanced).sort_values("WAPE")
print("\n===== ENHANCED RESULTS (sorted by WAPE) =====")
print(summary_enh.to_string(index=False))
display(summary_enh)

# Compare with previous tuning baseline
baseline_wape = {"xgboost_tuned": 43.34, "random_forest_tuned": 45.61}
print("\n===== IMPROVEMENT vs PREVIOUS TUNING =====")
for _, row in summary_enh.iterrows():
    model = row["model"]
    new_wape = row["WAPE"]
    old_wape = baseline_wape.get("xgboost_tuned", None)
    if old_wape:
        delta = old_wape - new_wape
        print(f"  {model}: WAPE {new_wape:.2f}% (improved by {delta:+.2f}pp vs {old_wape:.2f}%)")



===== ENHANCED RESULTS (sorted by WAPE) =====
           model      MAE      RMSE      WAPE  MAPE(>=5)  R2_norm
xgboost_enhanced 6.119282 12.148173 38.387733  56.730697 0.021681


,model,MAE,RMSE,WAPE,MAPE(>=5),R2_norm
0,xgboost_enhanced,6.119282,12.148173,38.387733,56.730697,0.021681



===== IMPROVEMENT vs PREVIOUS TUNING =====
  xgboost_enhanced: WAPE 38.39% (improved by +4.95pp vs 43.34%)


In [11]:
# Save results
from datetime import datetime
OUT_BASE_ROOT = "/user/data/results/demand_prediction/enhanced"
run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_dir = f"{OUT_BASE_ROOT}/run_{run_id}"

metrics_sdf = spark.createDataFrame(summary_enh)
metrics_sdf.write.mode("overwrite").parquet(f"{out_dir}/metrics")

cv_xgb_enh_model.bestModel.write().overwrite().save(f"{out_dir}/best_model_xgb_enhanced")

print(f"Saved to HDFS: {out_dir}")
print("Best enhanced model:", summary_enh.iloc[0]["model"])
print(f"Best WAPE: {summary_enh.iloc[0]['WAPE']:.2f}%")


26/05/08 15:13:29 WARN TaskSetManager: Stage 671 contains a task of very large size (1938 KiB). The maximum recommended task size is 1000 KiB.


Saved to HDFS: /user/data/results/demand_prediction/enhanced/run_20260508_151328
Best enhanced model: xgboost_enhanced
Best WAPE: 38.39%



- Thêm feature `hour_of_week` (168 giá trị) thay vì `hour` + `dow` riêng lẻ
- Thử LightGBM thay XGBoost, tốt hơn cho tabular data thưa